# Werewolf Transformer — Stage 1 population iteration (iPad / Colab)

128村 smoke run が完了した後に使います。
Google Drive 上の `pilot-001` をそのまま使い、**payoff計測 → meta-strategy → Village/Werewolf/Fox oracle** を1周だけ実行します。
途中で Colab が切れても、再度このノートブックを上から実行すれば自動 resume します。

設定はまだ保守的に `parallel_games=8`, inference cap 64 のままです。


In [ ]:
# 1) Google Drive を接続
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 2) 最新 main を取得して依存関係をインストール
%cd /content
!rm -rf Are-you-werewolf
!git clone --depth 1 https://github.com/dolphin23-jp/Are-you-werewolf.git
%cd /content/Are-you-werewolf/backend
!python -m pip install -q -e ".[rl,transformer]"


In [ ]:
# 3) GPU と smoke-run 保存物を確認
from pathlib import Path
import json
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU が有効ではありません。ランタイムのタイプを GPU に変更してください。')

RUN_ROOT = Path('/content/drive/MyDrive/werewolf-training/pilot-001')
POOL_DIR = RUN_ROOT / 'pool'
BOOTSTRAP = RUN_ROOT / 'bootstrap'
POPULATION = RUN_ROOT / 'population'

required = [POOL_DIR / 'manifest.json', BOOTSTRAP / 'run.npz', BOOTSTRAP / 'metrics.jsonl']
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise RuntimeError('128村 smoke run の保存物が見つかりません: ' + ', '.join(missing))

rows = [json.loads(line) for line in (BOOTSTRAP / 'metrics.jsonl').read_text().splitlines() if line.strip()]
if not rows or rows[-1].get('completed_episodes') != 128:
    raise RuntimeError('128村 smoke run が完了していません。先に smoke-run notebook を完了してください。')

print('GPU:', torch.cuda.get_device_name(0))
print('bootstrap completed:', rows[-1]['completed_episodes'])
print('population save dir:', POPULATION)


In [ ]:
# 4) Stage 1: population iteration を1周。既存stateがあれば自動resume。
import subprocess

POPULATION.mkdir(parents=True, exist_ok=True)
state_path = POPULATION / 'population.run.json'
log_path = POPULATION / 'stage1-console.log'

common = [
    'python', 'scripts/run_population_iterations_torch.py',
    '--pool-dir', str(POOL_DIR),
    '--run-dir', str(POPULATION),
    '--iterations', '1',
    '--device', 'auto',
]

if state_path.exists():
    cmd = common + ['--resume']
    print('既存の population state を検出したため resume します。')
else:
    cmd = common + [
        '--recent-policies', '3',
        '--games-per-profile', '3',
        '--extra-games', '8',
        '--oracle-episodes', '64',
        '--oracle-batch-size', '16',
        '--parallel-games', '8',
        '--inference-batch-size', '64',
        '--evaluation-seed', '1101',
        '--oracle-seed', '1201',
        '--opponent-seed', '1301',
    ]
    print('新しい Stage 1 population iteration を開始します。')

with log_path.open('a', encoding='utf-8') as log:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    code = proc.wait()
if code != 0:
    raise RuntimeError(f'population iteration failed with exit code {code}. 上の最後のエラーを ChatGPT に送ってください。')

print('\nSTAGE 1 COMPLETE')


In [ ]:
# 5) ChatGPT に送る Stage 1 結果を表示
import json

summary_path = POPULATION / 'iteration-0001' / 'summary.json'
log_path = POPULATION / 'stage1-console.log'
if not summary_path.exists():
    raise RuntimeError('summary.json がありません。直前のセルがまだ完了していないか、エラーで停止しています。')

summary = json.loads(summary_path.read_text())
print('===== POPULATION SUMMARY =====')
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('===== END POPULATION SUMMARY =====')

if log_path.exists():
    lines = [line for line in log_path.read_text().splitlines() if 'event=oracle_batch' in line or 'event=meta_solved' in line or 'event=iteration_completed' in line]
    print('\n===== LEARNING EVENTS (LAST 20) =====')
    print('\n'.join(lines[-20:]))
    print('===== END LEARNING EVENTS =====')

print('\n上の POPULATION SUMMARY と LEARNING EVENTS をこのチャットに貼ってください。')


## 中断した場合

Google Drive 上に outer run-state と oracle NPZ が残ります。
このノートブックを新しい Colab セッションで上から実行すれば、4番目のセルが自動的に `--resume` を選びます。

Stage 1 完了後はまだ数千村へ進まず、最後の summary / learning events を ChatGPT に送ってください。
